# 记忆路由（Memory Routing）

> **用一个智能调度器，把每一次记忆读写都导向正确的专用存储。**

想象一间大办公室，里面有几个分开的文件柜。一个柜子放着客户谈话记录，另一个放着公司政策，第三个放着逐步操作手册。有人要归档或取一份文件时，得先决定打开*哪个柜子*。这个决定就是路由。

随着智能体记忆系统变大，它会拆分成多个专用存储。你可能用一个**情景记忆**存储放对话历史，一个**语义记忆**存储放事实知识，一个**程序性记忆**存储放学到的技能。每种存储有自己的数据格式和检索策略。难点在于：智能体怎么知道某条信息该用哪个存储？

没有显式路由时，智能体会把所有东西塞进一个大存储里，失去专业化的好处。**记忆路由**的解法是在智能体和各存储之间加一层调度器。写入时，调度器对内容分类（这是事实、流程，还是经历？），再送到对应的存储。读取时，它分析查询意图，检索相关的一个或多个存储。多个存储都命中时，它合并结果并重排序。

**在本 notebook 中你将构建：**
1. 三个专用记忆存储（情景、语义、程序性）。
2. 一个由 LLM 驱动的路由器，按类型对内容和查询分类。
3. 把操作导向正确存储的写入和读取流程。
4. 针对跨存储模糊查询的回退策略。


## 核心概念

- **记忆类型分类（memory type classification）**：把进来的内容分成情景、语义、程序性等类别。分类器可以是一次 LLM 调用、关键词规则，或一个训练过的小模型。
- **情景记忆**：关于事件和经历的记忆。「昨天的会议」「用户上一次的请求」「下午 3 点发生的那次报错」。
- **语义记忆**：关于事实和通用知识的记忆。「API 速率限制是每分钟 1,000 次请求。」「Python 用缩进表示代码块。」
- **程序性记忆**：关于怎么做一件事的记忆。逐步说明、工作流、操作手册。「部署步骤：跑测试、开 PR、合并到 main。」
- **读取路由（read routing）**：分析查询意图，决定检索哪个（或哪些）存储。「昨天发生了什么？」路由到情景存储。「我们的退款政策是什么？」路由到语义存储。
- **写入路由（write routing）**：把新内容以正确的格式送到正确的存储。对话事件带时间戳，事实带实体标签。
- **存储注册表（store registry）**：可用记忆存储的目录。它把每种记忆类型映射到一个存储，附带描述和能力说明。
- **回退策略（fallback strategy）**：路由器无法自信分类时的处理方式。常见做法：检索所有存储并合并结果，或默认用一个通用存储。


## 架构

<p align="center">
  <img src="../../images/diagrams/17_memory_routing.svg" alt="记忆路由架构图" width="720"/>
</p>

**智能体**把每一次记忆操作（读或写）发给**记忆路由器**。路由器运行一个分类器，判断内容类型。写入时，它把内容送到匹配的专用存储：情景（对话和事件）、语义（事实和知识）或程序性（技能和工作流）。读取时，路由器根据查询意图检索一个或多个存储。结果经过**合并与重排序**（Merge & Re-rank）步骤，把来自不同存储的条目汇总打分。**存储注册表**告诉路由器每个存储里有什么、该怎么用。


## 准备工作

安装依赖并配置你的 API key。你需要把 `LLM_API_KEY` 写进 `.env` 文件，或设为环境变量。


In [ ]:
%pip install -q python-dotenv deepagents langchain-openai

导入 OpenAI SDK 和标准库辅助工具。API key 从 `.env` 文件加载。


In [ ]:
import os
import json
from enum import Enum
from dataclasses import dataclass, field
from datetime import datetime

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

### 记忆类型与记忆条目

先搭数据模型。`MemoryType` 枚举定义路由器可以分入的三个类别。`MemoryEntry` 保存内容、类型、时间戳和唯一 ID。


In [ ]:
class MemoryType(Enum):
    """路由器支持的几种记忆类别。"""
    EPISODIC = "episodic"       # 对话、会议、事件
    SEMANTIC = "semantic"       # 事实、定义、政策
    PROCEDURAL = "procedural"   # 逐步说明、工作流


@dataclass
class MemoryEntry:
    """存在某个记忆存储里的一条条目。"""
    content: str
    memory_type: MemoryType
    metadata: dict = field(default_factory=dict)
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    id: str = field(default_factory=lambda: os.urandom(8).hex())

    def __repr__(self) -> str:
        preview = self.content[:60] + ("..." if len(self.content) > 60 else "")
        return f"MemoryEntry(type={self.memory_type.value}, content={preview!r})"

### 记忆存储

每种存储只放一种类型的条目。`search` 方法用关键词重叠来找相关条目。生产环境中，你会把它换成向量检索（比较文本含义的数值表示）或数据库查询。为了教学透明，这里用关键词重叠。


In [ ]:
class MemoryStore:
    """存放一类记忆的专用存储。"""

    def __init__(self, name: str, memory_type: MemoryType, description: str):
        self.name = name
        self.memory_type = memory_type
        self.description = description
        self.entries: list[MemoryEntry] = []

    def add(self, entry: MemoryEntry) -> None:
        """向这个存储追加一条条目。"""
        self.entries.append(entry)

    def search(self, query: str, top_k: int = 5) -> list[MemoryEntry]:
        """按与查询的关键词重叠找出条目。"""
        query_words = set(query.lower().split())
        scored = []
        for entry in self.entries:
            content_words = set(entry.content.lower().split())
            overlap = len(query_words & content_words)
            if overlap > 0:
                scored.append((overlap, entry))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [entry for _, entry in scored[:top_k]]

    def list_all(self) -> list[MemoryEntry]:
        return list(self.entries)

    def __len__(self) -> int:
        return len(self.entries)

    def __repr__(self) -> str:
        return f"{self.name}Store({len(self.entries)} entries)"


现在为每种记忆类型各建一个存储。
每个存储有名称和描述。
路由器会根据这些描述决定进来的内容属于哪里。


In [ ]:
# 为每种记忆类型各建一个存储
episodic_store = MemoryStore(
    name="Episodic",
    memory_type=MemoryType.EPISODIC,
    description="Conversations, meetings, events, and personal experiences.",
)

semantic_store = MemoryStore(
    name="Semantic",
    memory_type=MemoryType.SEMANTIC,
    description="Facts, definitions, policies, and general knowledge.",
)

procedural_store = MemoryStore(
    name="Procedural",
    memory_type=MemoryType.PROCEDURAL,
    description="Step-by-step instructions, workflows, and how-to guides.",
)

stores = {
    MemoryType.EPISODIC: episodic_store,
    MemoryType.SEMANTIC: semantic_store,
    MemoryType.PROCEDURAL: procedural_store,
}

print("Stores created:", list(stores.values()))

### 记忆路由器

路由器是核心组件。它接下每一个读或写请求，问 LLM：「这是哪种记忆？」然后根据回答把操作导向正确的存储。

`classify` 方法向 LLM 发送一段短提示词。提示词列出每个存储及其描述，然后要求模型返回一个 JSON 数组，内容是匹配的类型名。写入时，路由器创建一个 `MemoryEntry` 并加进目标存储。读取时，它检索匹配的存储，返回合并后的列表。

我们还提供 `read_all_stores` 作为**回退**（fallback，即常规路径不确定时采用的默认行为）。查询含糊时，你可以检索所有存储，让结果自己说话。


In [ ]:
class MemoryRouter:
    """把记忆的读取和写入路由到正确的专用存储。"""

    def __init__(
        self,
        client: OpenAI,
        model: str,
        stores: dict[MemoryType, MemoryStore],
    ):
        self.client = client
        self.model = model
        self.stores = stores
        self.routing_log: list[dict] = []  # 记录每一次路由决策

    # ── 分类 ───────────────────────────────────────────
    def classify(self, text: str, operation: str) -> list[MemoryType]:
        """让 LLM 把文本分类为一种或多种记忆类型。"""
        store_lines = "\n".join(
            f"- {mt.value}: {store.description}"
            for mt, store in self.stores.items()
        )

        prompt = (
            f"Classify this {operation} into one or more memory types.\n\n"
            f"Available types:\n{store_lines}\n\n"
            f"Text: {text}\n\n"
            f"Return ONLY a JSON array of matching type names.\n"
            f'Example: ["episodic", "semantic"]\n'
            f"Do not include any other text."
        )

        response = llm.invoke([{"role": "user", "content": prompt}])

        raw = to_text(response.content).strip()
        # 如果模型用 markdown 代码围栏包住 JSON，就去掉围栏
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        type_names = json.loads(raw)
        result = [MemoryType(name) for name in type_names]

        # 记录每次路由决策，方便调试
        self.routing_log.append({
            "operation": operation,
            "text_preview": text[:80],
            "routed_to": [t.value for t in result],
        })

        return result


路由器还需要 `write` 和 `read` 方法。
`write` 对内容分类，存进匹配的存储。
`read` 对查询分类，检索匹配的存储，返回去重后的结果。


In [ ]:
def write(self, content: str, metadata: dict | None = None) -> list[str]:
    """给内容分类，并存入匹配的存储。"""
    types = self.classify(content, operation="write")
    metadata = metadata or {}

    destinations = []
    for memory_type in types:
        entry = MemoryEntry(
            content=content,
            memory_type=memory_type,
            metadata=metadata,
        )
        self.stores[memory_type].add(entry)
        destinations.append(memory_type.value)

    return destinations

# ── 读取路由 ─────────────────────────────────────────────
MemoryRouter.write = write

def read(self, query: str, top_k: int = 5) -> list[MemoryEntry]:
    """给查询分类，检索匹配的存储，返回合并结果。"""
    types = self.classify(query, operation="read")

    all_results = []
    for memory_type in types:
        results = self.stores[memory_type].search(query, top_k=top_k)
        all_results.extend(results)

    # 按条目 ID 去重（同一条内容可能存在于多个存储）
    seen = set()
    unique = []
    for entry in all_results:
        if entry.id not in seen:
            seen.add(entry.id)
            unique.append(entry)

    return unique[:top_k]
MemoryRouter.read = read

对于模糊查询，`read_all_stores` 会检索每个存储并合并结果。
这就是回退机制（分类器拿不准时的安全默认值）。
`get_stats` 返回每个存储的条目数，用于监控。


In [ ]:
def read_all_stores(self, query: str, top_k: int = 5) -> list[MemoryEntry]:
    """检索所有存储。路由器无法自信分类时使用。"""
    all_results = []
    for store in self.stores.values():
        all_results.extend(store.search(query, top_k=top_k))

    seen = set()
    unique = []
    for entry in all_results:
        if entry.id not in seen:
            seen.add(entry.id)
            unique.append(entry)

    return unique[:top_k]

# ── 统计 ────────────────────────────────────────────────────
MemoryRouter.read_all_stores = read_all_stores

def get_stats(self) -> dict[str, int]:
    """返回每个存储的条目数。"""
    return {mt.value: len(store) for mt, store in self.stores.items()}
MemoryRouter.get_stats = get_stats

## 运行示例

我们把路由器完整跑一遍。先存入六条不同类型的记忆，再查询它们。注意看路由器如何给每条内容分类，并把它送到正确的存储。


In [ ]:
router = MemoryRouter(client=client, model=MODEL, stores=stores)

# 六条记忆，覆盖全部三种类型
memories_to_store = [
    "In Monday's standup, Alice reported a 2-hour payment API outage over the weekend.",
    "Our API rate limit is 1,000 requests per minute per client.",
    "To deploy to production: run the test suite, open a pull request, get one approval, then merge to main.",
    "During the Q3 review, the team decided to migrate from PostgreSQL to CockroachDB.",
    "The company refund policy allows full refunds within 30 days of purchase.",
    "To reset a user password: verify identity with two security questions, generate a reset token, send the reset email.",
]

print("=== Writing Memories ===\n")
for content in memories_to_store:
    destinations = router.write(content)
    print(f"Content:  {content[:75]}...")
    print(f"Routed to: {destinations}\n")

print("Store counts:", router.get_stats())

现在开始查询。每个问题意图清晰，都对应一个存储。路由器先给查询分类，然后只检索相关的存储。


In [ ]:
queries = [
    "What happened at the standup meeting?",
    "What is our API rate limit?",
    "How do I deploy to production?",
]

print("=== Reading Memories ===\n")
for query in queries:
    results = router.read(query)
    print(f"Query: {query}")
    if results:
        for entry in results:
            print(f"  [{entry.memory_type.value}] {entry.content[:80]}")
    else:
        print("  (no results)")
    print()

### 查看路由日志

每一次路由决策都有记录。这份日志对调试被误路由的记忆很有价值，也能帮你持续改进分类器。


In [ ]:
print("=== Routing Log (all decisions so far) ===\n")
for i, entry in enumerate(router.routing_log):
    direction = "WRITE" if entry["operation"] == "write" else "READ "
    targets = ", ".join(entry["routed_to"])
    print(f"  {i+1:2d}. [{direction}] -> {targets}")
    print(f"      {entry['text_preview']}")
    print()

### 多存储查询与回退

有些查询是模糊的。「我们对数据库都知道些什么？」既可能涉及情景记忆（迁移决定），也可能涉及语义记忆（数据库事实）。路由器可能只把它归为一种类型，漏掉另一个存储的结果。

`read_all_stores` 方法是一种**扇出回退**（fan-out fallback）：它检索每个存储并合并全部结果。代价更高（每个存储各查一次），但保证你不会漏掉任何结果。


In [ ]:
ambiguous_query = "What do we know about the database migration?"
print(f"Query: {ambiguous_query}\n")

# 标准路由读取
routed_results = router.read(ambiguous_query)
print(f"Routed read ({len(routed_results)} results):")
for entry in routed_results:
    print(f"  [{entry.memory_type.value}] {entry.content[:80]}")

# 回退：扇出到所有存储
fallback_results = router.read_all_stores(ambiguous_query)
print(f"\nFallback read ({len(fallback_results)} results):")
for entry in fallback_results:
    print(f"  [{entry.memory_type.value}] {entry.content[:80]}")

## 权衡

### 记忆路由的适用场景

- **模块化。** 每个存储可以独立优化。你可以更换后端、调优检索、修改模式（数据格式），不用动其他存储。
- **检索精度。** 专用存储里只有一种类型的内容。需要过滤的噪声更少，搜索结果更相关。
- **可扩展性。** 加一种新记忆类型（比如存用户设置的「偏好」存储）只需新建一个存储并注册它，不用改现有存储，也不用改智能体的核心逻辑。

### 它什么时候会出问题

- **延迟开销。** 每次操作都要先做一次分类调用才能到达存储。用 LLM 分类器时，每次操作多出 200-500 毫秒。基于关键词或嵌入的分类器更快，但准确率更低。
- **误路由。** 分类器选错类型时，写入的内容进错存储，读取也就永远找不到它。从用户角度看，误路由的数据等于悄悄丢了。好的日志和回退策略能降低这种风险。
- **模糊内容。** 有些信息横跨多种类型。「昨天的会上，我们把退款期限定成了 14 天」既是情景记忆（一次会议事件），也是语义记忆（一条政策事实）。路由器必须支持多标签分类，这意味着同一条内容可能存放在多个存储里。
- **分类器成本。** 每次路由决策都消耗 token。高吞吐系统里，分类成本会累积。可以考虑缓存高频模式，或改用更小的本地模型。


## 延伸阅读

- Packer 等，["MemGPT: Towards LLMs as Operating Systems,"](https://arxiv.org/abs/2310.08560) 2023。展示了如何通过显式的记忆管理函数，在主上下文和外部存储之间路由记忆操作。
- Zhang 等，["A Survey on the Memory Mechanism of Large Language Model Based Agents,"](https://arxiv.org/abs/2404.13501) 2024。对智能体架构中的记忆类型（情景、语义、程序性）及其交互方式做了分类。
- Lewis 等，["Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks,"](https://arxiv.org/abs/2005.11401) 2020。RAG 的奠基论文。记忆路由把 RAG 模式推广到多个专用存储。
- Weston 等，["Memory Networks,"](https://arxiv.org/abs/1410.3916) 2015。早期工作：用带显式记忆模块的神经网络和基于注意力的路由访问记忆槽。
- Park 等，["Generative Agents: Interactive Simulacra of Human Behavior,"](https://arxiv.org/abs/2304.03442) 2023。使用多条独立的记忆流（观察、反思、计划），检索时综合新近度、重要性和相关性打分。


*← 上一篇：[16 - 自我反思记忆](../16_self_reflection_memory/) · 下一篇：[18 - 时序记忆](../18_temporal_memory/) →*


## 🧪 动手练习

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：路由准确率基准
创建 20 条带人工标注类型的测试记忆（情景、语义、程序性）。逐条送入 `MemoryRouter.classify()`，统计分类对错。算出每种类型的准确率，找出路由器最容易混淆的类型。

### 挑战 2：单存储检索与全量回退的延迟对比
用 `read()`（定向单存储查询）和 `read_all_stores()`（检索每个存储）各跑 30 次查询。测量每次查询的实际耗时。算出两者的平均延迟差，以及回退模式能找到而定向查询漏掉结果的查询占比。

### 挑战 3：四存储架构
在情景和语义存储之外，加第四个 `MemoryStore` 来存放程序性知识。更新 `classify()` 的提示词以支持新类型。把 20 条混合记忆送进四存储系统，验证每条都放对了位置。这直接扩展了 11 程序性记忆中的程序性存储。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--17-memory-routing--memory-routing)